### RAG Pipelines = Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/_z/45d7lx5s18970c_3xcbsw3vr0000gn/T/ipykernel_4578/3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the drectory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for file in pdf_files:
        print(f"\nProcessing: {file.name}")
        try:
            loader = PyPDFLoader(str(file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages") 

        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

## Process all PDFs in the data directory
all_pdfs_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: embeddings.pdf
Loaded 4 pages

Processing: attention.pdf
Loaded 4 pages

Processing: object_detection.pdf
Loaded 4 pages

Processing: proposal.pdf
Loaded 3 pages

Total documents loaded: 15


In [3]:
all_pdfs_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='Vector Embeddings for Semantic Search:\nConcepts, Models and Evaluation\nMariana Costa Thiago Ribeiro\nDepartment of Computer Science, Federal Technology University\nJune 2024\nAbstract\nEmbeddings are dense numerical vectors that represent the meaning of words, sentences,\nimages, or docume

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')

    # Show example of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f'Content: {split_docs[0].page_content[:200]}...')
        print(f'Metadata: {split_docs[0].metadata}')

    return split_docs


In [5]:
chunks = split_documents(all_pdfs_documents)
chunks

Split 15 documents into 39 chunks

Example chunk:
Content: Vector Embeddings for Semantic Search:
Concepts, Models and Evaluation
Mariana Costa Thiago Ribeiro
Department of Computer Science, Federal Technology University
June 2024
Abstract
Embeddings are dens...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-17T20:22:05-03:00', 'author': 'Mariana Costa, Thiago Ribeiro', 'title': 'Vector Embeddings for Semantic Search: Concepts, Models and Evaluation', 'subject': 'Information Retrieval', 'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG', 'moddate': '2026-11-17T20:22:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '../data/pdf/embeddings.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'embeddings.pdf', 'file_type': 'pdf'}, page_content='Vector Embeddings for Semantic Search:\nConcepts, Models and Evaluation\nMariana Costa Thiago Ribeiro\nDepartment of Computer Science, Federal Technology University\nJune 2024\nAbstract\nEmbeddings are dense numerical vectors that represent the meaning of words, sentences,\nimages, or docume

### embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/rafaelrocha/Documents/PROJETOS/PJRAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 01:44:17.682000 4578 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [7]:
class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """"
        Generate embedding for a list of texts

        Args: 
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9674.90it/s]


Model loaded successfully. Embedding dimension: 384


## VectorStore

In [8]:
class VectorStore:
    """Manages document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """"
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF documents embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embedding to the vectore store
        
        Args:
            documents: list of LangChain documents
            embeddings: Corresponding embedding for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
## Convert the text to embeddings

texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## Store in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 39 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.15it/s]

Generated embeddings with shape: (39, 384)
Adding 39 documents to vector store...
Successfully added 39 documents to vector store
Total documents in collection: 39


## Retriever Pipeline From VectorStore

In [20]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """"
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """"
        Retrieve relevant documents for a query

        Args:
            query: Ther search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")    

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embedding_manager)

In [11]:
rag_retriever

In [12]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_08190f9b_20',
  'document': 'foundation of the Transformer. Its quadratic cost remains the central engineering challenge, and\na large body of work now focuses on making attention cheaper without losing accuracy. For\npractitioners, understanding how attention consumes context is directly useful when designing\nprompts, choosing chunk sizes, and building retrieval pipelines.\n4',
  'metadata': {'page': 3,
   'doc_index': 20,
   'content_length': 347,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'moddate': '2026-11-17T20:22:04-03:00',
   'file_type': 'pdf',
   'source': '../data/pdf/attention.pdf',
   'trapped': '/False',
   'source_file': 'attention.pdf',
   'author': 'Lucas Andrade, Helena Prado',
   'keywords': 'attention, transformers, self-attention, multi-head attention, NLP',
   'producer': 'pdfTeX-1.40.25',
   'creationdate': '2026-11-17T20:22:04-03:00',

In [14]:
rag_retriever.retrieve("Embeddings compress meaning into a fixed number of dimensions", score_threshold=-1)


Retrieving documents for query: 'Embeddings compress meaning into a fixed number of dimensions'
Top K: 5, Score threshold: -1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_743b88bf_9',
  'document': '7 Limitations\nEmbeddings compress meaning into a fixed number of dimensions, so fine distinctions can be lost.\nThey may struggle with negation (“hotels that are not near the beach”), numbers, and very domain-\nspecific vocabulary. They also inherit biases present in their training data. Finally, re-embedding\nan entire collection is required whenever the embedding model is changed, which has real cost for\nlarge datasets.\n8 Conclusion\nEmbeddings turn the problem of understanding meaning into the problem of measuring distance\nbetween vectors. Contrastively trained sentence models, combined with approximate nearest neigh-\nbor indexes, make semantic search fast and practical. For RAG systems, the quality of retrieval\ndepends as much on chunking, metadata, and evaluation as on the choice of model itself.\n4',
  'metadata': {'keywords': 'embeddings, semantic search, vector database, cosine similarity, RAG',
   'producer': 'pdfTeX-1.40.25',
   

## Integration Vectordb Context pipeline With LLM output

In [ ]:
## Simple RAG pipeline with LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM (set your GRO_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question"

    ## Generate the answer using GROQ LLM
    prompt = """Use the following context to answer the question concisely.
        Answer in plain text only. Never use LaTeX or Markdown math.
        Write formulas in plain notation, e.g. softmax(Q*K^T / sqrt(d_k)) * V.

        Context:
        {context}

        Question: {query}

        Answer:"""


    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [ ]:
answer=rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'oq é mecanismo de atencao?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question


## Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is attention mechanism?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])